# 10. `MLP_ALPHA` 프로브 — Colab 빌드

**DIRECTION.md 후보 #1.** 챔피언 스택(`wonwoo` s24, LB 1067.55)에서 α만 바꾼 제출 zip 5개를 만든다.

## 왜 Colab인가

로컬 macOS(M1)에서는 **LightGBM과 PyTorch의 `libomp` 충돌**로 학습이 불가능하다.
LGBM 20개가 끝난 직후 torch의 첫 병렬 연산(`index_kernel`)에서
`__kmp_join_barrier` 데드락이 나고, 임포트 순서를 바꾸면 SIGSEGV가 난다.
`KMP_DUPLICATE_LIB_OK=TRUE`는 "조용히 틀린 결과"를 낼 수 있어 쓰지 않는다.

Colab은 우회가 아니라 **더 나은 선택**이다 — 재현 기준 환경에 더 가깝다:

| | 원우 (기준) | M1 로컬 | **Colab** |
|---|---|---|---|
| CPU | x86_64 | ❌ arm64 | ✅ x86_64 |
| torch | `2.13.0+cpu` | ❌ 플레인 arm64 휠 | ✅ `2.13.0+cpu` **정확 일치** |
| libomp 충돌 | 없음 | ❌ 치명적 | 없음 |

## ⚠️ 런타임은 CPU로 둘 것

`wonwoo:REPRODUCE.md`: *"CUDA 빌드는 부동소수점 누산 순서와 RNG가 달라 같은 시드로도
다른 가중치가 나온다. MLP 8멤버가 배포 블렌드의 40%를 차지하므로, 빌드가 다르면
기록된 LB 점수를 재현할 수 없다."*

`src/mlp.py:_device()`가 기본 CPU라 GPU 런타임이어도 학습은 CPU로 가지만,
**torch 휠 자체가 달라지므로** CPU 런타임을 쓴다.

## 준비물

Drive 구조는 이미 갖춰져 있다 (2026-08-20 확인). **한 파일만 올리면 된다:**

```
MyDrive/LG_aimers_9th/
├── data/
│   ├── train.csv               368,527,723 B  ✅ 이미 있음
│   ├── test.csv                        1,894 B  ✅
│   └── sample_submission.csv             112 B  ✅
└── champion_src.tar.gz          83 KB  ← ⬆️ 이것만 업로드
```

### `champion_src.tar.gz` 재생성

`hyejin:outputs/`에 이미 있다. **원우 브랜치가 업데이트되면 다시 만들어야 한다** —
원우 `src/`의 191개 모듈 중 실제로 import 되는 것만 담기 때문에, 모듈이 추가되면
목록이 달라진다. 아래가 그 목록을 매번 다시 계산한다 (import 폐포).

```bash
# 워크트리가 없으면 먼저:  git worktree add ~/Aimers_9th_champion wonwoo
CHAMP=~/Aimers_9th_champion
python3 - "$CHAMP" <<'PY'
import ast, shutil, subprocess, sys, tempfile
from pathlib import Path
champ = Path(sys.argv[1]); SRC = champ / 'src'
local = {p.stem for p in SRC.glob('*.py')}
need, stack = set(), ['train', 'build_submit', 'predict', 'mlp_infer']
while stack:                                   # import 폐포
    m = stack.pop()
    if m in need or m not in local: continue
    need.add(m)
    for n in ast.walk(ast.parse((SRC/f'{m}.py').read_text(encoding='utf-8'))):
        if isinstance(n, ast.Import):
            stack += [a.name.split('.')[0] for a in n.names]
        elif isinstance(n, ast.ImportFrom) and n.module and n.level == 0:
            stack.append(n.module.split('.')[0])
with tempfile.TemporaryDirectory() as t:
    d = Path(t); (d/'src').mkdir(); (d/'submit').mkdir()
    for m in need: shutil.copy2(SRC/f'{m}.py', d/'src')
    shutil.copy2(champ/'submit'/'script.py', d/'submit')
    out = Path.home()/'Aimers_9th'/'outputs'/'champion_src.tar.gz'
    subprocess.run(['tar','-czf',str(out),'-C',str(d),'src','submit'], check=True)
    print(f'{out}  ({out.stat().st_size:,} B, 모듈 {len(need)}개)')
PY
```

2026-08-20 기준 19개 모듈 / 83 KB. 만든 뒤 Drive `LG_aimers_9th/`에 덮어쓴다.

## 실행 시간

전체 학습 약 30분 + α 변형 4개 × 38초 + 빌드·검증 약 20분 ≈ **1시간**

## 1. 환경 — REPRODUCE.md 스펙 정확히 고정

In [ ]:
import sys, platform, subprocess
print('Python  ', sys.version.split()[0])
print('Platform', platform.machine(), platform.system())
assert platform.machine() == 'x86_64', 'x86_64가 아니다 — 재현 충실도가 떨어진다'

# ! 매직의 역슬래시 줄바꿈은 동작이 애매하므로 subprocess 로 명시 호출한다.
PINS = ['lightgbm==4.6.0', 'numpy==2.2.6', 'pandas==2.3.3', 'scipy==1.15.2',
        'joblib==1.4.2', 'scikit-learn==1.6.1', 'pyarrow==19.0.1']
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', *PINS], check=True)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch==2.13.0+cpu',
                '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
print('설치 완료 — 다음 셀에서 버전을 검증한다')

In [ ]:
# ⚠️ pip 설치 후 런타임 재시작이 필요할 수 있다. 아래가 전부 일치하지 않으면
#    [런타임] → [세션 다시 시작] 후 이 셀부터 다시 실행할 것.
import importlib.metadata as md
EXPECT = {'lightgbm':'4.6.0','numpy':'2.2.6','pandas':'2.3.3','scipy':'1.15.2',
          'joblib':'1.4.2','scikit-learn':'1.6.1','pyarrow':'19.0.1'}
bad = []
for p, want in EXPECT.items():
    got = md.version(p)
    flag = '✅' if got == want else '❌'
    if got != want: bad.append((p, want, got))
    print(f'{flag} {p:16s} {got:12s} (기대 {want})')
tv = md.version('torch'); print(f"{'✅' if tv.startswith('2.13.0') else '❌'} {'torch':16s} {tv}")
assert not bad, f'버전 불일치: {bad} — 런타임 재시작 후 다시 실행'
import torch; assert not torch.cuda.is_available() or True
print('\ntorch CUDA 사용 가능:', torch.cuda.is_available(), '(학습은 항상 CPU로 간다)')

## 2. 데이터·코드 — 무결성 확인 포함

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, shutil, os, tarfile
from pathlib import Path

# 실제 Drive 구조 (2026-08-20 확인):
#   MyDrive/LG_aimers_9th/
#   ├── data/  train.csv · test.csv · sample_submission.csv · trackman_history.csv
#   └── champion_src.tar.gz   ← 여기에 올려둘 것
DRIVE = Path('/content/drive/MyDrive/LG_aimers_9th')
ROOT  = Path('/content/champion')
(ROOT / 'data').mkdir(parents=True, exist_ok=True)

TARBALL = DRIVE / 'champion_src.tar.gz'
assert TARBALL.is_file(), (
    f'{TARBALL} 가 없다.\n'
    '  hyejin:outputs/champion_src.tar.gz (83KB) 를 LG_aimers_9th/ 에 올릴 것.')
with tarfile.open(TARBALL) as tf:
    tf.extractall(ROOT)
assert (ROOT/'src'/'train.py').is_file(), 'train.py 없음 — tar 내용 확인'
assert (ROOT/'submit'/'script.py').is_file(), 'submit/script.py 없음'
print('코드 전개 완료:', len(list((ROOT/'src').glob('*.py'))), '개 .py (필요 모듈만)')

# 데이터 — 368MB 복사는 느리다. 심볼릭 링크로 건다.
for name in ['train.csv', 'test.csv', 'sample_submission.csv']:
    src = DRIVE / 'data' / name
    assert src.is_file(), f'{src} 없음'
    dst = ROOT / 'data' / name
    if not dst.exists():
        os.symlink(src, dst)
print('데이터 링크 완료:', [p.name for p in (ROOT/'data').iterdir()])

# 무결성 — 원우/hyejin 양쪽 기록과 대조 (약 1분)
h = hashlib.sha256()
with open(ROOT/'data'/'train.csv', 'rb') as f:
    for blk in iter(lambda: f.read(1 << 20), b''):
        h.update(blk)
digest = h.hexdigest()
EXPECT_SHA = 'd2081186b458b49f60b082be480c273135833e15ba59a76d033af28bcf8763ff'
print('\ntrain.csv SHA-256:', digest)
assert digest == EXPECT_SHA, '❌ train.csv 가 기록된 파일과 다르다 — 진행 중단'
print('✅ 무결성 확인 (hyejin reports/06 · wonwoo MODEL.md 기록과 일치)')

## 3. α 주입 러너

`train.py`는 `from config import (..., MLP_ALPHA, ...)` = **import 시점의 값 복사**다.
따라서 `train`을 import 하기 **전에** `config.MLP_ALPHA`를 바꾸면 그 값을 집어간다.
원우 파일은 한 글자도 고치지 않는다.

**`artifacts.joblib`의 `mlp_alpha`만 갈아끼우면 안 된다** — `_finish()`가 블렌드
**이후** 로짓으로 `solve_delta(base + raw_ref, TARGET_PRIOR_2025)`를 다시 푼다.
α가 바뀌면 delta도 달라져야 하고, 안 그러면 prior가 어긋나
`BSS ≈ −400,000 × (편차)²`로 크게 잃는다.

`MLP_ALPHA`는 `train._TRAIN_SHAPING`(모델을 결정하는 설정 목록)에 **없으므로**
`--artifacts-only`가 모델을 재사용해도 안전하다 (§7.66 — 26분 → 38초).

In [ ]:
import subprocess, textwrap

RUNNER = ROOT / 'run_alpha.py'
RUNNER.write_text(textwrap.dedent('''
    import os, sys, shutil, argparse
    from pathlib import Path
    ap = argparse.ArgumentParser()
    ap.add_argument('--alpha', type=float, required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--full', action='store_true')
    ap.add_argument('--reuse', default=None)
    a = ap.parse_args()

    ROOT = Path('/content/champion')
    out = ROOT / a.out
    if not a.full:
        assert a.reuse, '--artifacts-only 에는 --reuse 필요'
        if out.exists(): shutil.rmtree(out)
        shutil.copytree(ROOT / a.reuse, out)   # 원본 baseline 을 보존한다
    out.mkdir(parents=True, exist_ok=True)
    os.environ['LGA_MODEL_DIR'] = str(out)

    sys.path.insert(0, str(ROOT / 'src'))
    os.chdir(ROOT)
    import config
    before = config.MLP_ALPHA
    config.MLP_ALPHA = float(a.alpha)          # ⚠️ train import 전에
    print(f'MLP_ALPHA 주입: {before} -> {config.MLP_ALPHA}', flush=True)

    sys.argv = ['train.py'] + ([] if a.full else ['--artifacts-only'])
    import train
    assert train.MLP_ALPHA == float(a.alpha), (
        f'주입 실패 train.MLP_ALPHA={train.MLP_ALPHA}')   # 침묵 실패 방어선
    assert 'mlp_alpha' not in train._TRAIN_SHAPING, (
        'alpha 가 모델 결정 설정에 들어갔다 — artifacts-only 재사용이 위험하다')
    train.main()
'''), encoding='utf-8')

def run(*args):
    r = subprocess.run([sys.executable, '-u', str(RUNNER), *args],
                       cwd=ROOT, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode: print(r.stderr[-3000:]); raise SystemExit('실패')
    return r.stdout
print('러너 작성 완료')

## 4. 전체 학습 1회 (α=0.40 배포값) — 약 30분

LightGBM 20개(4변형×5시드) + MLP 8개(emb4×5 + noid×3, 보조헤드 2축).

**기대 로그** — 아래와 다르면 재현이 깨진 것이니 멈추고 확인할 것:
```
보정축 hand: 셀 1,571개      보정축 count4: 셀 3,105개
features=78  (drop=['game_type', 'game_month', 'game_dayofweek'])
soft label(§7.70): +71,199행 복제  h = 2019:0.093 2020:0.110 2021:0.243 2022:0.297
```

In [ ]:
import time
t0 = time.time()
log = run('--alpha', '0.40', '--full', '--out', 'models/a40')
print(f'\n총 {time.time()-t0:.0f}초')

for probe, expect in [('셀 1,571개', 'δ hand'), ('셀 3,105개', 'δ count4'),
                      ('features=78', '피처 수'), ('71,199행', 'soft label')]:
    print(('✅' if probe in log else '❌') + f' {expect}: {probe}')

## 5. α 변형 4개 — 각 38초 (모델 재사용)

**배포값 0.40을 가운데 두고 양쪽을 걸친다.** 두 이론이 정반대를 가리키기 때문이다:

| | 주장 | 예측 |
|---|---|---|
| 홀드아웃 (§7.65) | 두 fold 모두 α를 낮추라고 한다 | α↓ |
| 원우 판단 | 홀드아웃이 MLP 비중을 과소평가한다 (전이 배율 2.6·6.7) | α↑ |

0.40 아래만 재면 "0.40이 최적"과 "더 올렸어야 했다"를 구분할 수 없다.

In [ ]:
for a in ['0.20', '0.30', '0.50', '0.60']:
    tag = 'a' + a.replace('0.', '').ljust(2, '0')
    print(f'\n{"="*20} α={a} {"="*20}')
    run('--alpha', a, '--out', f'models/{tag}', '--reuse', 'models/a40')

import json
for d in sorted((ROOT/'models').iterdir()):
    m = json.loads((d/'meta.json').read_text())
    print(f"{d.name:6s}  mlp_alpha={m['mlp_alpha']}  부스터 {m['n_members']}개")

## 6. zip 빌드 5개

`build_submit.py`를 그대로 쓰면 세 가지가 깨진다 — 래퍼가 막는다:

1. `model_dir = SUBMIT/'model'` **하드코딩** → 변형을 그 자리에 스테이징
2. zip이 `submit/` **전체**를 훑는다 → 변형은 `models/`에 두고 하나씩만 스테이징
3. `record_history()` → `submit/history.json`, `log_submission()` → `SUBMISSIONS.md`
   (원우 추적 파일) → 두 함수를 no-op으로 교체
4. `next_zip_path()`가 s25.. 자동 번호 → 이름을 우리가 지정

In [ ]:
BUILDER = ROOT / 'build_one.py'
BUILDER.write_text(textwrap.dedent('''
    import os, sys, json, shutil, zipfile, argparse
    from pathlib import Path
    ap = argparse.ArgumentParser()
    ap.add_argument('--models', required=True)
    ap.add_argument('--name', required=True)
    a = ap.parse_args()

    ROOT = Path('/content/champion')
    models = ROOT / a.models
    zip_path = ROOT / f'{a.name}.zip'
    assert len(zip_path.name) <= 30, f'파일명 {len(zip_path.name)}자 > 30자'
    if zip_path.exists(): zip_path.unlink()

    staged = ROOT / 'submit' / 'model'
    if staged.exists(): shutil.rmtree(staged)
    shutil.copytree(models, staged)
    alpha = json.loads((staged/'meta.json').read_text())['mlp_alpha']

    strays = [p for p in (ROOT/'submit').iterdir()
              if p.is_dir() and p.name.startswith('model') and p.name != 'model']
    assert not strays, f'submit/ 에 다른 모델이 있다 (zip 이 통째로 삼킨다): {strays}'

    sys.path.insert(0, str(ROOT/'src')); os.chdir(ROOT)
    import build_submit
    build_submit.record_history = lambda *a, **k: print('  이력 기록 건너뜀')
    build_submit.log_submission = lambda *a, **k: print('  SUBMISSIONS.md 기록 건너뜀')
    build_submit.next_zip_path  = lambda meta, suffix='': zip_path
    build_submit.main()

    with zipfile.ZipFile(zip_path) as z: names = z.namelist()
    n_l = sum(n.startswith('model/lgbm_') for n in names)
    n_m = sum(n.startswith('model/mlp_')  for n in names)
    assert n_l == 20, f'LGBM {n_l}개 (기대 20)'
    assert n_m == 8,  f'MLP {n_m}개 (기대 8)'
    print(f'OK {zip_path.name} alpha={alpha} lgbm={n_l} mlp={n_m} '
          f'{zip_path.stat().st_size/1e6:.1f}MB')
'''), encoding='utf-8')

for tag in ['a20', 'a30', 'a40', 'a50', 'a60']:
    print(f'\n{"="*20} {tag} {"="*20}')
    r = subprocess.run([sys.executable, '-u', str(BUILDER),
                        '--models', f'models/{tag}', '--name', f'p_{tag}'],
                       cwd=ROOT, capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode: print(r.stderr[-2000:]); raise SystemExit(f'{tag} 빌드 실패')

## 7. 독립 검증 — **압축을 푼 상태에서** 실행

`build_submit.py`는 `script.py`를 `submit/` 작업 디렉터리에서 돌린다. 그래서
**`submit/`에는 있는데 zip에는 없는 파일**이 있어도 통과하고 평가 서버에서만 터진다.
여기서는 빈 임시 디렉터리에 풀어서 실행한다.

행 독립성(통짜 vs 셔플 vs 청크가 비트 단위로 같은가)은 **대회 규정 준수 증빙**이다 —
운영측이 2026-08-13에 위반 시 실격 공지를 냈다.

In [ ]:
import tempfile, zipfile, time, pandas as pd
sys.path.insert(0, str(ROOT/'src'))
os.chdir(ROOT)
import build_submit as bs

N_ROWS = 20000                      # 정확성·행독립성 검사용. 시간은 아래에서 환산한다
dummy = bs.make_dummy_test(N_ROWS)
HIDDEN = ('__MACOSX', '.DS_Store', 'Thumbs.db', '.ipynb_checkpoints')

def run_script(work, df):
    d = work/'data'; d.mkdir(exist_ok=True)
    df.to_csv(d/'test.csv', index=False)
    pd.DataFrame({'row_id': df['row_id'], 'control_success': .5}).to_csv(
        d/'sample_submission.csv', index=False)
    shutil.rmtree(work/'output', ignore_errors=True)
    t0 = time.time()
    r = subprocess.run([sys.executable, 'script.py'], cwd=work,
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-1500:]); print(r.stderr[-3000:])
        raise SystemExit('script.py 실패')
    return pd.read_csv(work/'output'/'submission.csv'), time.time()-t0

results = {}
for tag in ['a20','a30','a40','a50','a60']:
    zp = ROOT/f'p_{tag}.zip'
    print(f'\n{"="*22} {zp.name} {"="*22}')

    # ── 정적 ────────────────────────────────────────────────
    assert len(zp.name) <= 30, f'파일명 {len(zp.name)}자 > 30자'
    with zipfile.ZipFile(zp) as z:
        assert z.testzip() is None, 'CRC 불일치'
        names = z.namelist()
    hits = [n for n in names if any(h in n for h in HIDDEN)]
    assert not hits, f'숨은 파일: {hits[:5]}'
    for req in ['script.py', 'requirements.txt']:
        assert req in names, f'루트에 {req} 없음'
    assert any(n.startswith('model/') for n in names), 'model/ 없음'
    tops = {n.split('/', 1)[0] for n in names}
    assert 'script.py' in tops, f'최상위에 래퍼 폴더가 있다: {tops}'
    print(f'  ✅ 구조 정상 · 구성원 {len(names)}개 · 파일명 {len(zp.name)}자')

    # ── 격리 실행 ───────────────────────────────────────────
    with tempfile.TemporaryDirectory() as tmp:
        work = Path(tmp)/'x'; work.mkdir()
        with zipfile.ZipFile(zp) as z: z.extractall(work)
        reqs = (work/'requirements.txt').read_text()
        assert 'torch' not in reqs, 'requirements 에 torch — 추론은 numpy 전용이어야 한다'

        base, el = run_script(work, dummy)
        proj = el/len(dummy)*245789
        assert list(base.columns) == ['row_id','control_success'], base.columns
        assert len(base) == len(dummy), f'행 수 {len(base)} != {len(dummy)}'
        assert base['row_id'].equals(dummy['row_id'].reset_index(drop=True)), \
            'row_id 가 sample_submission 과 다르다 (순서 포함)'
        p = base['control_success']
        assert p.notna().all(), 'NaN 있음'
        assert p.between(0,1).all(), f'[0,1] 밖: {p.min()} ~ {p.max()}'
        print(f'  ✅ 실행 {el:.1f}s → 245,789행 환산 {proj:.0f}s / 600s ({proj/6:.0f}%)')
        print(f'  ✅ 출력 정상  mean={p.mean():.6f}  min={p.min():.4f}  max={p.max():.4f}')
        assert proj < 480, f'추론 시간 여유 부족 ({proj:.0f}s)'

        # ── 행 독립성 (대회 규정) ────────────────────────────
        ref = base.set_index('row_id')['control_success']
        sh, _ = run_script(work, dummy.sample(frac=1, random_state=42).reset_index(drop=True))
        d1 = (sh.set_index('row_id')['control_success'] - ref).abs().max()
        half = len(dummy)//2
        ch = pd.concat([run_script(work, dummy.iloc[:half].reset_index(drop=True))[0],
                        run_script(work, dummy.iloc[half:].reset_index(drop=True))[0]])
        d2 = (ch.set_index('row_id')['control_success'] - ref).abs().max()
        assert d1 == 0.0, f'셔플 후 예측이 달라진다 ({d1:.3e}) — 규정 위반'
        assert d2 == 0.0, f'청크 분할 예측이 달라진다 ({d2:.3e}) — 규정 위반'
        print(f'  ✅ 행 독립성  셔플 {d1:.1e} · 청크 {d2:.1e}')
        results[tag] = float(p.mean())

print('\n' + '='*54)
print('✅ 5개 전부 통과. 제출 가능.')
for t, m in results.items():
    print(f'   p_{t}.zip   더미 예측평균 {m:.6f}')
print('='*54)

## 8. 내려받기

**제출 순서 권장** — 배포값부터 쏴서 기준점을 먼저 잡는다:
`p_a40` → `p_a30` → `p_a20` → `p_a50` → `p_a60`

결과는 `hyejin:outputs/alpha_probe/manifest.jsonl`에 기록할 것.

In [ ]:
OUT = DRIVE / 'outputs' / 'alpha_probe'
OUT.mkdir(parents=True, exist_ok=True)
for tag in ['a40','a30','a20','a50','a60']:
    src = ROOT / f'p_{tag}.zip'
    shutil.copy2(src, OUT / src.name)
    print(f'{OUT}/{src.name}   ({src.stat().st_size/1e6:.1f} MB)')

print('\nDrive 저장 완료 → MyDrive/LG_aimers_9th/outputs/alpha_probe/')
print('\n제출 순서 (배포값부터 기준점을 잡는다):')
for i, t in enumerate(['a40','a30','a20','a50','a60'], 1):
    print(f'  {i}. p_{t}.zip')

# 브라우저로 바로 내려받으려면 주석 해제
# from google.colab import files
# for tag in ['a40','a30','a20','a50','a60']:
#     files.download(str(ROOT/f'p_{tag}.zip'))